# Ejemplo End-to-End: Pandas con Titanic

En este notebook haremos un flujo más completo y realista de análisis con pandas usando el dataset del Titanic.

La idea no es solo mirar algunas columnas, sino recorrer un mini proceso end-to-end:

1. **Carga de datos**
2. **Exploración inicial**
3. **Calidad de datos**
4. **Limpieza**
5. **Feature engineering**
6. **Análisis con `groupby`, `pivot_table` y ranking**
7. **Segmentación de pasajeros**
8. **Visualización básica**
9. **Tabla ejecutiva final**

Este ejemplo mezcla varias operaciones de pandas en un caso más cercano a un análisis real.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print('Shape inicial:', df.shape)
display(df.head())

## 1. Exploración inicial

Primero entendemos rápidamente qué trae la base.

In [ ]:
print('Columnas:')
print(df.columns.tolist())

print('\nTipos de dato:')
display(df.dtypes.to_frame('dtype'))

print('\nResumen numérico:')
display(df.describe())

print('\nResumen categórico:')
display(df.describe(include='object'))

## 2. Calidad de datos

Antes de transformar, revisamos nulos, duplicados y algunas validaciones básicas.

In [ ]:
quality = pd.DataFrame({
    'nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(2),
    'n_unicos': df.nunique(dropna=False)
}).sort_values('pct_nulos', ascending=False)

display(quality)

print('Filas duplicadas:', df.duplicated().sum())
print('Fare negativos:', (df['Fare'] < 0).sum())
print('Age negativos:', (df['Age'] < 0).sum())

## 3. Limpieza de datos

Vamos a hacer una limpieza un poco más pensada:

- `Cabin` tiene demasiados nulos, así que no la usaremos completa.
- `Age` la imputamos con la mediana por sexo y clase.
- `Embarked` la completamos con la moda.
- Creamos una copia limpia para no perder el original.

In [ ]:
df_clean = df.copy()

# Extraemos solo la letra de la cabina cuando exista
df_clean['CabinDeck'] = df_clean['Cabin'].str[0]

# Imputación de Age usando mediana por Sex y Pclass
df_clean['Age'] = df_clean.groupby(['Sex', 'Pclass'])['Age'].transform(lambda s: s.fillna(s.median()))

# Si aún quedara algún nulo, usamos la mediana global
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Embarked con la moda
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Columnas menos útiles para este análisis
df_clean = df_clean.drop(columns=['Cabin', 'Ticket'])

print('Shape después de limpieza:', df_clean.shape)
print('\nNulos después de limpieza:')
display(df_clean.isna().sum().to_frame('nulos'))

## 4. Feature engineering

Ahora creamos variables más analíticas.

In [ ]:
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = np.where(df_clean['FamilySize'] == 1, 1, 0)
df_clean['FarePerPerson'] = df_clean['Fare'] / df_clean['FamilySize']
df_clean['AgeBand'] = pd.cut(
    df_clean['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Niño', 'Adolescente', 'Adulto joven', 'Adulto', 'Mayor'],
    include_lowest=True
)
df_clean['FareBand'] = pd.qcut(
    df_clean['Fare'],
    q=4,
    labels=['Bajo', 'Medio-bajo', 'Medio-alto', 'Alto'],
    duplicates='drop'
)
df_clean['Title'] = df_clean['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
df_clean['Title'] = df_clean['Title'].replace({
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Don': 'Rare',
    'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare', 'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
})

display(df_clean.head())

## 5. Análisis descriptivo más útil

Aquí ya podemos responder preguntas más interesantes.

In [ ]:
print('Tasa global de supervivencia:')
print(round(df_clean['Survived'].mean(), 4))

survival_summary = df_clean.groupby(['Sex', 'Pclass']).agg(
    pasajeros=('PassengerId', 'count'),
    tasa_supervivencia=('Survived', 'mean'),
    edad_promedio=('Age', 'mean'),
    tarifa_promedio=('Fare', 'mean')
).reset_index()

survival_summary['tasa_supervivencia'] = survival_summary['tasa_supervivencia'].round(3)
survival_summary['edad_promedio'] = survival_summary['edad_promedio'].round(1)
survival_summary['tarifa_promedio'] = survival_summary['tarifa_promedio'].round(2)

display(survival_summary.sort_values('tasa_supervivencia', ascending=False))

## 6. `pivot_table` para resumir mejor

Las tablas dinámicas ayudan mucho para presentar resultados.

In [ ]:
pivot_survival = pd.pivot_table(
    df_clean,
    values='Survived',
    index='Sex',
    columns='Pclass',
    aggfunc='mean'
).round(3)

display(pivot_survival)

In [ ]:
pivot_fare = pd.pivot_table(
    df_clean,
    values='Fare',
    index='Embarked',
    columns='Pclass',
    aggfunc=['mean', 'median']
).round(2)

display(pivot_fare)

## 7. Ranking dentro de grupos con `transform`

Queremos comparar a cada pasajero contra su propio segmento.

In [ ]:
df_clean['fare_rank_in_class'] = df_clean.groupby('Pclass')['Fare'].rank(method='dense', ascending=False)
df_clean['fare_vs_class_avg'] = df_clean['Fare'] / df_clean.groupby('Pclass')['Fare'].transform('mean')
df_clean['age_vs_sex_avg'] = df_clean['Age'] / df_clean.groupby('Sex')['Age'].transform('mean')

cols_show = ['PassengerId', 'Name', 'Pclass', 'Sex', 'Fare', 'fare_rank_in_class', 'fare_vs_class_avg', 'age_vs_sex_avg']
display(df_clean[cols_show].head(10))

## 8. Segmentación de pasajeros

Construimos perfiles simples para entender mejor la población.

In [ ]:
conditions = [
    (df_clean['Age'] <= 12),
    (df_clean['Sex'] == 'female') & (df_clean['Pclass'] <= 2),
    (df_clean['Sex'] == 'male') & (df_clean['Pclass'] == 3),
]
choices = ['Niño/a', 'Mujer clase alta/media', 'Hombre clase baja']

df_clean['segmento_pasajero'] = np.select(conditions, choices, default='Otros')

segment_summary = df_clean.groupby('segmento_pasajero').agg(
    pasajeros=('PassengerId', 'count'),
    tasa_supervivencia=('Survived', 'mean'),
    edad_promedio=('Age', 'mean'),
    tarifa_promedio=('Fare', 'mean')
).sort_values('tasa_supervivencia', ascending=False)

display(segment_summary.round(2))

## 9. Filtros analíticos con `query`

Esto deja el código más legible en algunos análisis.

In [ ]:
casos_interesantes = df_clean.query(
    "Sex == 'female' and Pclass == 3 and Survived == 1"
)[['PassengerId', 'Name', 'Age', 'Fare', 'Embarked', 'Title']]

display(casos_interesantes.head(10))

## 10. Top combinaciones con más supervivencia

Ahora un resumen más ejecutivo.

In [ ]:
top_segments = (
    df_clean
    .groupby(['Sex', 'Pclass', 'Embarked', 'AgeBand'], observed=False)
    .agg(
        pasajeros=('PassengerId', 'count'),
        tasa_supervivencia=('Survived', 'mean')
    )
    .reset_index()
)

top_segments = top_segments[top_segments['pasajeros'] >= 10]
top_segments = top_segments.sort_values(['tasa_supervivencia', 'pasajeros'], ascending=[False, False])

display(top_segments.head(15).round(3))

## 11. Visualizaciones rápidas

No buscamos hacer un notebook de visualización, pero sí dejar 2 o 3 gráficos útiles.

In [ ]:
plt.figure(figsize=(8, 5))
df_clean.groupby('Pclass')['Survived'].mean().plot(kind='bar')
plt.title('Tasa de supervivencia por clase')
plt.ylabel('Tasa de supervivencia')
plt.xlabel('Pclass')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar')
plt.title('Tasa de supervivencia por sexo')
plt.ylabel('Tasa de supervivencia')
plt.xlabel('Sexo')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
df_clean.groupby('AgeBand', observed=False)['Survived'].mean().plot(kind='bar')
plt.title('Tasa de supervivencia por grupo de edad')
plt.ylabel('Tasa de supervivencia')
plt.xlabel('Grupo de edad')
plt.show()

## 12. Tabla final ejecutiva

Creamos una tabla final que un analista sí podría exportar o compartir.

In [ ]:
executive_table = (
    df_clean
    .groupby(['Pclass', 'Sex', 'AgeBand'], observed=False)
    .agg(
        pasajeros=('PassengerId', 'count'),
        sobrevivientes=('Survived', 'sum'),
        tasa_supervivencia=('Survived', 'mean'),
        tarifa_promedio=('Fare', 'mean'),
        edad_promedio=('Age', 'mean')
    )
    .reset_index()
)

executive_table['tasa_supervivencia'] = executive_table['tasa_supervivencia'].round(3)
executive_table['tarifa_promedio'] = executive_table['tarifa_promedio'].round(2)
executive_table['edad_promedio'] = executive_table['edad_promedio'].round(1)

display(executive_table.sort_values(['tasa_supervivencia', 'pasajeros'], ascending=[False, False]).head(20))

## 13. Conclusiones

Con este ejemplo ya no solo hicimos un análisis básico. También vimos cómo:

- revisar calidad de datos,
- imputar valores faltantes,
- crear variables derivadas,
- usar `groupby`, `transform`, `pivot_table` y `query`,
- construir tablas ejecutivas finales.

Ese flujo se parece mucho más a un caso real de trabajo con pandas.